# Análisis V2 - Regresión Logística: Campaña Agrícola

## 📌 Contexto

Este notebook documenta la **segunda versión (V2)** del modelo de Regresión Logística para clasificar la **campaña agrícola** (verano vs invierno).

### Ventaja clave sobre KNN
La Regresión Logística **sí acepta `class_weight='balanced'`**, lo que permite penalizar más los errores en la clase minoritaria (invierno).

## 1. Metodología V2

**Mejoras respecto a V1:**
1. `class_weight='balanced'` → compensa el desbalanceo 9:1
2. `GridSearchCV` sobre `C` (0.01-100), `penalty` (L2), `solver` (lbfgs, liblinear)
3. `StratifiedKFold` (5-fold)
4. Scoring = F1
5. Métricas completas + coeficientes interpretables

## 2. Resultados

### 2.1 Mejores hiperparámetros

| Parámetro | Valor |
|---|---|
| C | 10 |
| penalty | l2 |
| solver | lbfgs |
| F1 (validación cruzada) | **0.3959** |

### 2.2 Métricas en test

| Métrica | Valor |
|---|---|
| Accuracy | 0.8281 |
| Precision (Invierno) | 0.2456 |
| Recall (Invierno) | **0.3333** |
| F1 (Invierno) | **0.2828** |
| ROC-AUC | **0.6572** |

### 2.3 Classification Report

```
              precision    recall  f1-score   support

      Verano       0.92      0.88      0.90       371
    Invierno       0.25      0.33      0.28        42

    accuracy                           0.83       413
   macro avg       0.58      0.61      0.59       413
weighted avg       0.85      0.83      0.84       413
```

## 3. Interpretación de coeficientes

```
SUPERFICIE : +3.7712
PRODUCCION : -5.3447
```

### 📖 Lectura agronómica

El modelo calcula: `log_odds(invierno) = 3.77·Superficie - 5.34·Produccion + b`

- **Superficie (+)**: parcelas más grandes → más probable invierno.
- **Producción (-)**: mayor producción → más probable verano.
- **PRODUCCION pesa más** (|5.34| > |3.77|): es el mejor discriminador.

### 🧠 Interpretación agronómica
- **Verano**: parcelas más chicas pero muy productivas (mejores condiciones climáticas).
- **Invierno**: parcelas más grandes pero con menor rendimiento (clima adverso, siembras extensivas).

Esto tiene sentido: en invierno se apuesta por más hectáreas para compensar el bajo rendimiento por Ha.

## 4. Visualización

![Regresión Logística V2](../03_imagenes/27_regresion_logistica_v2.png)

**Observaciones:**
- Frontera de decisión: **lineal** (por eso la Regresión Logística es un clasificador lineal).
- Curva ROC mejor que KNN (AUC 0.657 vs 0.615).
- Recall de invierno mejor que KNN (0.33 vs 0.19), gracias a `class_weight='balanced'`.

## 5. Comparación con KNN V2

| Métrica | KNN V2 | Reg. Logística V2 | Ganador |
|---|---|---|---|
| Accuracy | **0.8644** | 0.8281 | KNN |
| Precision (Inv) | **0.2667** | 0.2456 | KNN |
| Recall (Inv) | 0.1905 | **0.3333** | Reg. Log. |
| F1 (Inv) | 0.2222 | **0.2828** | Reg. Log. |
| ROC-AUC | 0.6147 | **0.6572** | Reg. Log. |
| F1 (CV) | 0.3628 | **0.3959** | Reg. Log. |

### 🏆 Veredicto
Aunque KNN gana en accuracy, la **Regresión Logística es superior en las métricas que realmente importan** cuando hay desbalanceo: F1, Recall y ROC-AUC.

Además, la Reg. Log. es **interpretable** (coeficientes) mientras que KNN es una caja negra.

## 6. Conclusiones y próximos pasos

### ✅ Aprendizajes
1. **`class_weight='balanced'` funciona**: subió el recall de invierno de 0.19 → 0.33.
2. **A mayor C (menos regularización), mejor F1**: el modelo se beneficia de ajustar más libremente con estos datos.
3. **Coeficientes interpretables** permiten contar la historia agronómica.

### ⚠️ Limitaciones
- F1 de invierno aún es bajo (0.28) → el modelo falla mucho en la clase rara.
- ROC-AUC de 0.657 apenas supera el azar.
- Con solo 2 features hay poco margen de mejora.

### 🔜 Próximos pasos (V3)
1. **Ajustar umbral de decisión** (bajar de 0.5 a ~0.3) → debería subir recall.
2. **Agregar `RENDIMIENTO (Kg/Ha)`** como tercera feature.
3. **OneHotEncoding de `DEPARTAMENTO`** (Santa Cruz, Beni, La Paz, etc.).
4. **SMOTE** para generar inviernos sintéticos.
5. **Probar modelos no lineales** (Random Forest, Gradient Boosting).